In [1]:
import numpy as np

# 임시 데이터 생성
rng = np.random.default_rng(0)
temp = rng.normal(180, 3, 300) # 평균 180, 표준편차 3을 따르는 정상 온도 데이터 300개를 생성

# 온도가 180 을 넘으면 온도가 오름(180:0, 181:5.05, ...)
press = 5 + 0.05 * (temp - 180) 
press += rng.normal(0, 0.03, 300)   # 미세한 공정 노이즈.
X = np.c_[temp, press] # 온도와 압력을 X 에 담는다.(300,2)

In [5]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# X 내 온도와 압력을 평균 0, 표준편차 1 로 표준화 한다.
sc = StandardScaler().fit(X)
Xs = sc.transform(X)

# 표준화된 데이터 Xs를 기준으로 최대 2개 주성분 축(PC1, PC2)을 찾습니다.
pca = PCA(n_components=2).fit(Xs)

print('설명 분산 비율 :', pca.explained_variance_ratio_.round(3))   # PC1 이 대부분

설명 분산 비율 : [0.991 0.009]


In [9]:
# 01_01 에서 사용했던 이상치
anomaly = [184.0, 4.82]

# 정상 데이터 Xs를 기준으로 핵심 방향인 제1주성분(PC1) 1개 축만 남기는 PCA 모델을 학습
pca1 = PCA(n_components=1).fit(Xs)  # 2차원 → 1차원으로 압축

def restore(v):
    # 입려받은 값 v 를 압축 : pca1.transform(v)
    # pca1.inverse_transform() : 원래 2차원(표준화 공간)으로 복원
    return pca1.inverse_transform(pca1.transform(v))

In [ ]:
# 정상 데이터(Xs) 와 복원된 값의 차이를 제곱
gap = (Xs - restore(Xs)) ** 2 # 제곱을 통해 음수를 없애고, 오차를 증폭 시킨다.
# print(gap) # [온도,압력]

# 각 row 단위(axis=1)로 평균을 내어 정상 데이터의 재구성 오차(MSE) 배열을 구한다.
err_normal = (gap).mean(axis=1)
# print(err_normal)

In [ ]:

# 테스트할 이상치 anomaly를 2차원 리스트 [anomaly]로 감싼 뒤, 앞서 학습한 StandardScaler(sc)로 정상 데이터와 동일한 기준(평균 0, 표준편차 1)으로 변환합니다.
a = sc.transform([anomaly])

# 변환된 이상치 a를 recon(a)로 압축·복원한 뒤, 원래 a와의 제곱 오차 평균을 계산해 이상치의 재구성 오차를 구합니다.
gap = (a - restore(a)) ** 2
err_anom = (gap).mean()

In [ ]:
print(f'정상 평균 오차 {err_normal.mean():.3f} , 정상 최대 {err_normal.max():.3f} , 이상 {err_anom:.3f}')